In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/Internship Dataset.csv")

In [ ]:
df_clean = df.copy()

In [ ]:
missing_values = df_clean.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64


In [ ]:
duplicate_rows = df_clean.duplicated().sum()

print("Total duplicate rows:", duplicate_rows)

Total duplicate rows: 0


In [ ]:
print("Arrival time examples:")
print(df_clean["Arrival_time"].head(10))

print("\nDeparture time examples:")
print(df_clean["Departure_Time"].head(10))

Arrival time examples:
0    00:00:00
1    11:06:00
2    11:28:00
3    12:10:00
4    00:00:00
5    21:04:00
6    21:26:00
7    22:25:00
8    19:40:00
9    20:18:00
Name: Arrival_time, dtype: str

Departure time examples:
0    10:25:00
1    11:08:00
2    11:30:00
3    00:00:00
4    20:30:00
5    21:06:00
6    21:28:00
7    00:00:00
8    19:40:00
9    20:20:00
Name: Departure_Time, dtype: str


In [ ]:
df_clean["Arrival_time"] = pd.to_datetime(
    df_clean["Arrival_time"],
    format="%H:%M:%S"
).dt.time

df_clean["Departure_Time"] = pd.to_datetime(
    df_clean["Departure_Time"],
    format="%H:%M:%S"
).dt.time

In [ ]:
print(df_clean[["Arrival_time", "Departure_Time"]].dtypes)

Arrival_time      object
Departure_Time    object
dtype: object


In [ ]:
print(type(df_clean["Arrival_time"].iloc[0]))
print(type(df_clean["Departure_Time"].iloc[0]))

<class 'datetime.time'>
<class 'datetime.time'>


In [ ]:
train_data = df_clean.groupby("Train_No").agg(
    Start_Station=("Station_Name", "first"),
    End_Station=("Station_Name", "last"),
    Start_Departure=("Departure_Time", "first"),
    End_Arrival=("Arrival_time", "last")
).reset_index()

train_data.head(10)

,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival
0,107,SAWANTWADI R,MADGOAN JN.,10:25:00,12:10:00
1,108,MADGOAN JN.,SAWANTWADI R,20:30:00,22:25:00
2,128,MADGOAN JN.,CHHATRAPATI,19:40:00,17:45:00
3,290,DELHI-SAFDAR,DELHI-SAFDAR,18:30:00,02:30:00
4,401,AURANGABAD,VARANASI JN.,21:30:00,10:00:00
5,421,LUCKNOW JN.,SHRI MATA VA,23:00:00,08:00:00
6,422,SHRI MATA VA,LUCKNOW JN.,11:45:00,14:30:00
7,477,SIRSA,SIRSA,12:10:00,15:20:00
8,502,RAJENDRANAGA,AMBALA CANTT,10:30:00,09:30:00
9,504,PATNA JN.,BATHINDA JN,09:00:00,10:00:00


In [ ]:
cross_midnight = (
    train_data["End_Arrival"] < train_data["Start_Departure"]
).sum()

print("Journeys where arrival time is earlier than departure time:",
      cross_midnight)

print("Total train journeys:", len(train_data))

Journeys where arrival time is earlier than departure time: 1922
Total train journeys: 11113


In [ ]:
start_datetime = pd.to_datetime(
    train_data["Start_Departure"].astype(str),
    format="%H:%M:%S"
)

end_datetime = pd.to_datetime(
    train_data["End_Arrival"].astype(str),
    format="%H:%M:%S"
)

# Add one day when the journey crosses midnight
end_datetime = end_datetime.where(
    end_datetime >= start_datetime,
    end_datetime + pd.Timedelta(days=1)
)

train_data["Journey_Duration"] = end_datetime - start_datetime

train_data.head(10)

,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival,Journey_Duration
0,107,SAWANTWADI R,MADGOAN JN.,10:25:00,12:10:00,0 days 01:45:00
1,108,MADGOAN JN.,SAWANTWADI R,20:30:00,22:25:00,0 days 01:55:00
2,128,MADGOAN JN.,CHHATRAPATI,19:40:00,17:45:00,0 days 22:05:00
3,290,DELHI-SAFDAR,DELHI-SAFDAR,18:30:00,02:30:00,0 days 08:00:00
4,401,AURANGABAD,VARANASI JN.,21:30:00,10:00:00,0 days 12:30:00
5,421,LUCKNOW JN.,SHRI MATA VA,23:00:00,08:00:00,0 days 09:00:00
6,422,SHRI MATA VA,LUCKNOW JN.,11:45:00,14:30:00,0 days 02:45:00
7,477,SIRSA,SIRSA,12:10:00,15:20:00,0 days 03:10:00
8,502,RAJENDRANAGA,AMBALA CANTT,10:30:00,09:30:00,0 days 23:00:00
9,504,PATNA JN.,BATHINDA JN,09:00:00,10:00:00,0 days 01:00:00


In [ ]:
train_data["Journey_Duration"].describe()

count                     11113
mean     0 days 04:36:00.809862
std      0 days 05:21:10.016554
min             0 days 00:00:00
25%             0 days 01:01:00
50%             0 days 02:12:00
75%             0 days 06:05:00
max             0 days 23:55:00
Name: Journey_Duration, dtype: object

In [ ]:
zero_duration = train_data[
    train_data["Journey_Duration"] == pd.Timedelta(0)
]

print("Zero-duration journeys:", len(zero_duration))

zero_duration[
    [
        "Train_No",
        "Start_Station",
        "End_Station",
        "Start_Departure",
        "End_Arrival",
        "Journey_Duration"
    ]
].head(20)

Zero-duration journeys: 6


,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival,Journey_Duration
991,12617,ERNAKULAM. J,HAZRAT NIZAM,13:15:00,13:15:00,0 days
1205,12851,BILASPUR JN.,CHENNAI CENT,08:55:00,08:55:00,0 days
2022,16318,SHRI MATA VA,KANNIYAKUMAR,21:55:00,21:55:00,0 days
2327,18233,INDORE BG,BILASPUR JN.,17:15:00,17:15:00,0 days
2381,18477,PURI,HARIDWAR JN,20:55:00,20:55:00,0 days
2804,22633,TRIVANDRUM C,HAZRAT NIZAM,14:15:00,14:15:00,0 days


In [ ]:
df_clean[df_clean["Train_No"] == 12617][
    [
        "Train_No",
        "Station_Code",
        "Station_Name",
        "Arrival_time",
        "Departure_Time",
        "Distance"
    ]
]

,Train_No,Station_Code,Station_Name,Arrival_time,Departure_Time,Distance
17378,12617,ERS,ERNAKULAM. J,13:15:00,13:15:00,0
17379,12617,AWY,ALWAYE,13:33:00,13:35:00,19
17380,12617,TCR,TRICHUR,14:17:00,14:20:00,74
17381,12617,SRR,SHORANUR JN.,15:20:00,15:25:00,107
17382,12617,PTB,PATTAMBI,15:38:00,15:40:00,118
17383,12617,KTU,KUTTIPPURAM,15:58:00,16:00:00,136
17384,12617,TIR,TIRUR,16:18:00,16:20:00,151
17385,12617,PGI,PARPANANGADI,16:33:00,16:35:00,167
17386,12617,FK,FEROK,16:53:00,16:55:00,182
17387,12617,CLT,CALICUT,17:10:00,17:15:00,192


In [ ]:
test_train = df_clean[df_clean["Train_No"] == 12617].copy()

test_train["Arrival_seconds"] = (
    test_train["Arrival_time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

test_train[
    ["Station_Name", "Arrival_time", "Arrival_seconds"]
].tail(10)

,Station_Name,Arrival_time,Arrival_seconds
17416,ITARSI,00:50:00,3000
17417,BHOPAL,02:40:00,9600
17418,BINA,04:40:00,16800
17419,JHANSI JN,06:45:00,24300
17420,GWALIOR JN,08:05:00,29100
17421,MORENA,08:35:00,30900
17422,AGRA CANTT,10:10:00,36600
17423,MATHURA JN.,11:00:00,39600
17424,FARIDABAD,12:39:00,45540
17425,HAZRAT NIZAM,13:15:00,47700


In [ ]:
arrival_seconds = (
    df_clean["Arrival_time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

arrival_previous = arrival_seconds.groupby(df_clean["Train_No"]).shift(1)

midnight_jumps = (
    (arrival_seconds < arrival_previous)
).sum()

print("Arrival-time backward jumps:", midnight_jumps)

Arrival-time backward jumps: 3113


In [ ]:
departure_seconds = (
    df_clean["Departure_Time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

departure_previous = departure_seconds.groupby(
    df_clean["Train_No"]
).shift(1)

departure_midnight_jumps = (
    departure_seconds < departure_previous
).sum()

print("Departure-time backward jumps:", departure_midnight_jumps)

Departure-time backward jumps: 5065


In [ ]:
print("Original station records:", len(df_clean))

print("Train-level records:", len(train_data))

print("Unique trains:", df_clean["Train_No"].nunique())

Original station records: 186074
Train-level records: 11113
Unique trains: 11113


In [ ]:
final_distance = df_clean.groupby("Train_No")["Distance"].last()

print(final_distance.head(10))

Train_No
107      78
108      83
128     978
290    2694
401    1618
421    1276
422    1277
477    2616
502    1206
504    1313
Name: Distance, dtype: int64


In [ ]:
distance_order_check = df_clean.groupby("Train_No")["Distance"].apply(
    lambda x: x.is_monotonic_increasing
)

print("Trains with non-decreasing Distance:",
      distance_order_check.sum())

print("Total trains:", len(distance_order_check))

Trains with non-decreasing Distance: 11113
Total trains: 11113


In [ ]:
train_data["Total_Distance"] = (
    df_clean.groupby("Train_No")["Distance"].last().values
)

train_data.head(10)

,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival,Journey_Duration,Total_Distance
0,107,SAWANTWADI R,MADGOAN JN.,10:25:00,12:10:00,0 days 01:45:00,78
1,108,MADGOAN JN.,SAWANTWADI R,20:30:00,22:25:00,0 days 01:55:00,83
2,128,MADGOAN JN.,CHHATRAPATI,19:40:00,17:45:00,0 days 22:05:00,978
3,290,DELHI-SAFDAR,DELHI-SAFDAR,18:30:00,02:30:00,0 days 08:00:00,2694
4,401,AURANGABAD,VARANASI JN.,21:30:00,10:00:00,0 days 12:30:00,1618
5,421,LUCKNOW JN.,SHRI MATA VA,23:00:00,08:00:00,0 days 09:00:00,1276
6,422,SHRI MATA VA,LUCKNOW JN.,11:45:00,14:30:00,0 days 02:45:00,1277
7,477,SIRSA,SIRSA,12:10:00,15:20:00,0 days 03:10:00,2616
8,502,RAJENDRANAGA,AMBALA CANTT,10:30:00,09:30:00,0 days 23:00:00,1206
9,504,PATNA JN.,BATHINDA JN,09:00:00,10:00:00,0 days 01:00:00,1313


In [ ]:
train_data["Total_Distance"].isna().sum()

np.int64(0)

In [ ]:
train_data["Number_of_Stops"] = (
    df_clean.groupby("Train_No")["Station_Name"].count().values
)

train_data.head(10)

,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival,Journey_Duration,Total_Distance,Number_of_Stops
0,107,SAWANTWADI R,MADGOAN JN.,10:25:00,12:10:00,0 days 01:45:00,78,4
1,108,MADGOAN JN.,SAWANTWADI R,20:30:00,22:25:00,0 days 01:55:00,83,4
2,128,MADGOAN JN.,CHHATRAPATI,19:40:00,17:45:00,0 days 22:05:00,978,22
3,290,DELHI-SAFDAR,DELHI-SAFDAR,18:30:00,02:30:00,0 days 08:00:00,2694,14
4,401,AURANGABAD,VARANASI JN.,21:30:00,10:00:00,0 days 12:30:00,1618,12
5,421,LUCKNOW JN.,SHRI MATA VA,23:00:00,08:00:00,0 days 09:00:00,1276,5
6,422,SHRI MATA VA,LUCKNOW JN.,11:45:00,14:30:00,0 days 02:45:00,1277,5
7,477,SIRSA,SIRSA,12:10:00,15:20:00,0 days 03:10:00,2616,14
8,502,RAJENDRANAGA,AMBALA CANTT,10:30:00,09:30:00,0 days 23:00:00,1206,9
9,504,PATNA JN.,BATHINDA JN,09:00:00,10:00:00,0 days 01:00:00,1313,3


In [ ]:
train_data["Number_of_Stops"].describe()

count    11113.000000
mean        16.743814
std         12.993123
min          2.000000
25%          8.000000
50%         15.000000
75%         22.000000
max        118.000000
Name: Number_of_Stops, dtype: float64

In [ ]:
train_data.columns

Index(['Train_No', 'Start_Station', 'End_Station', 'Start_Departure',
       'End_Arrival', 'Journey_Duration', 'Total_Distance', 'Number_of_Stops'],
      dtype='str')

In [ ]:
train_data.isna().sum()

Train_No            0
Start_Station       0
End_Station         0
Start_Departure     0
End_Arrival         0
Journey_Duration    0
Total_Distance      0
Number_of_Stops     0
dtype: int64

In [ ]:
train_data["Train_No"].duplicated().sum()

np.int64(0)

In [ ]:
train_data["Journey_Duration"].describe()

count                     11113
mean     0 days 04:36:00.809862
std      0 days 05:21:10.016554
min             0 days 00:00:00
25%             0 days 01:01:00
50%             0 days 02:12:00
75%             0 days 06:05:00
max             0 days 23:55:00
Name: Journey_Duration, dtype: object

In [ ]:
train_data[train_data["Journey_Duration"] == pd.Timedelta(0)][
    ["Train_No", "Start_Station", "End_Station",
     "Start_Departure", "End_Arrival", "Total_Distance"]
].head(20)

,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival,Total_Distance
991,12617,ERNAKULAM. J,HAZRAT NIZAM,13:15:00,13:15:00,2751
1205,12851,BILASPUR JN.,CHENNAI CENT,08:55:00,08:55:00,1412
2022,16318,SHRI MATA VA,KANNIYAKUMAR,21:55:00,21:55:00,3765
2327,18233,INDORE BG,BILASPUR JN.,17:15:00,17:15:00,1008
2381,18477,PURI,HARIDWAR JN,20:55:00,20:55:00,2347
2804,22633,TRIVANDRUM C,HAZRAT NIZAM,14:15:00,14:15:00,2841


In [ ]:
id="m7k2p1"
df_clean["Arrival_seconds"] = (
    df_clean["Arrival_time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

df_clean["Departure_seconds"] = (
    df_clean["Departure_Time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

df_clean[
    ["Train_No", "Station_Name", "Arrival_time", "Departure_Time",
     "Arrival_seconds", "Departure_seconds"]
].head(10)

,Train_No,Station_Name,Arrival_time,Departure_Time,Arrival_seconds,Departure_seconds
0,107,SAWANTWADI R,00:00:00,10:25:00,0,37500
1,107,THIVIM,11:06:00,11:08:00,39960,40080
2,107,KARMALI,11:28:00,11:30:00,41280,41400
3,107,MADGOAN JN.,12:10:00,00:00:00,43800,0
4,108,MADGOAN JN.,00:00:00,20:30:00,0,73800
5,108,KARMALI,21:04:00,21:06:00,75840,75960
6,108,THIVIM,21:26:00,21:28:00,77160,77280
7,108,SAWANTWADI R,22:25:00,00:00:00,80700,0
8,128,MADGOAN JN.,19:40:00,19:40:00,70800,70800
9,128,KARMALI,20:18:00,20:20:00,73080,73200


In [ ]:
test_train = df_clean[df_clean["Train_No"] == 12617].copy()

test_train["Time_seconds"] = test_train["Arrival_seconds"]

test_train["Previous_seconds"] = test_train["Time_seconds"].shift(1)

test_train["Midnight_crossing"] = (
    test_train["Time_seconds"] < test_train["Previous_seconds"]
)

test_train[
    ["Station_Name", "Arrival_time", "Arrival_seconds",
     "Previous_seconds", "Midnight_crossing"]
].head(20)

,Station_Name,Arrival_time,Arrival_seconds,Previous_seconds,Midnight_crossing
17378,ERNAKULAM. J,13:15:00,47700,NaN,False
17379,ALWAYE,13:33:00,48780,47700.0,False
17380,TRICHUR,14:17:00,51420,48780.0,False
17381,SHORANUR JN.,15:20:00,55200,51420.0,False
17382,PATTAMBI,15:38:00,56280,55200.0,False
17383,KUTTIPPURAM,15:58:00,57480,56280.0,False
17384,TIRUR,16:18:00,58680,57480.0,False
17385,PARPANANGADI,16:33:00,59580,58680.0,False
17386,FEROK,16:53:00,60780,59580.0,False
17387,CALICUT,17:10:00,61800,60780.0,False


In [ ]:
test_train[
    test_train["Midnight_crossing"]
][
    ["Station_Name", "Arrival_time", "Arrival_seconds",
     "Previous_seconds", "Midnight_crossing"]
]

,Station_Name,Arrival_time,Arrival_seconds,Previous_seconds,Midnight_crossing
17400,BHATKAL,00:16:00,960,84960.0,True
17416,ITARSI,00:50:00,3000,81000.0,True


In [ ]:
test_train["Day_Offset"] = test_train["Midnight_crossing"].cumsum()

test_train["Elapsed_Arrival_Seconds"] = (
    test_train["Arrival_seconds"]
    + test_train["Day_Offset"] * 24 * 60 * 60
)

test_train[
    ["Station_Name", "Arrival_time", "Midnight_crossing",
     "Day_Offset", "Elapsed_Arrival_Seconds"]
].iloc[18:25]

,Station_Name,Arrival_time,Midnight_crossing,Day_Offset,Elapsed_Arrival_Seconds
17396,KASARAGOD,20:28:00,False,0,73680
17397,MANGALORE JN,21:25:00,False,0,77100
17398,UDUPI,23:04:00,False,0,83040
17399,KUNDAPURA,23:36:00,False,0,84960
17400,BHATKAL,00:16:00,True,1,87360
17401,KUMTA,01:16:00,False,1,90960
17402,KAWR,01:54:00,False,1,93240


In [ ]:
test_train[
    test_train["Midnight_crossing"]
][
    ["Station_Name", "Arrival_time",
     "Day_Offset", "Elapsed_Arrival_Seconds"]
]

,Station_Name,Arrival_time,Day_Offset,Elapsed_Arrival_Seconds
17400,BHATKAL,00:16:00,1,87360
17416,ITARSI,00:50:00,2,175800


In [ ]:
test_train.tail(5)[
    ["Station_Name", "Arrival_time", "Departure_Time",
     "Day_Offset", "Elapsed_Arrival_Seconds"]
]

,Station_Name,Arrival_time,Departure_Time,Day_Offset,Elapsed_Arrival_Seconds
17421,MORENA,08:35:00,08:37:00,2,203700
17422,AGRA CANTT,10:10:00,10:15:00,2,209400
17423,MATHURA JN.,11:00:00,11:03:00,2,212400
17424,FARIDABAD,12:39:00,12:41:00,2,218340
17425,HAZRAT NIZAM,13:15:00,13:15:00,2,220500


In [ ]:
start_seconds = (
    test_train["Departure_seconds"].iloc[0]
)

final_elapsed_arrival = (
    test_train["Elapsed_Arrival_Seconds"].iloc[-1]
)

corrected_duration_seconds = (
    final_elapsed_arrival - start_seconds
)

corrected_duration = pd.to_timedelta(
    corrected_duration_seconds,
    unit="s"
)

corrected_duration

Timedelta('2 days 00:00:00')

In [ ]:
train_data.loc[
    train_data["Train_No"] == 12617,
    ["Train_No", "Journey_Duration"]
]

,Train_No,Journey_Duration
991,12617,0 days


In [ ]:
train_data = train_data.drop(columns=["Journey_Duration"])

train_data.head(3)

,Train_No,Start_Station,End_Station,Start_Departure,End_Arrival,Total_Distance,Number_of_Stops
0,107,SAWANTWADI R,MADGOAN JN.,10:25:00,12:10:00,78,4
1,108,MADGOAN JN.,SAWANTWADI R,20:30:00,22:25:00,83,4
2,128,MADGOAN JN.,CHHATRAPATI,19:40:00,17:45:00,978,22


In [ ]:
def calculate_journey_duration(group):
    group = group.copy()

    # Detect midnight crossings using arrival times
    previous_arrival = group["Arrival_seconds"].shift(1)

    group["Midnight_Crossing"] = (
        group["Arrival_seconds"] < previous_arrival
    )

    # Count cumulative midnight crossings
    group["Day_Offset"] = group["Midnight_Crossing"].cumsum()

    # Convert arrival time into a continuous elapsed timeline
    group["Elapsed_Arrival_Seconds"] = (
        group["Arrival_seconds"]
        + group["Day_Offset"] * 24 * 60 * 60
    )

    # First departure → final arrival
    start_departure = group["Departure_seconds"].iloc[0]
    final_arrival = group["Elapsed_Arrival_Seconds"].iloc[-1]

    duration_seconds = final_arrival - start_departure

    return pd.Series({
        "Journey_Duration": pd.to_timedelta(
            duration_seconds, unit="s"
        )
    })


journey_durations = (
    df_clean.groupby("Train_No", sort=False)
    .apply(calculate_journey_duration)
    .reset_index()
)

journey_durations.head()

,Train_No,Journey_Duration
0,107,0 days 01:45:00
1,108,0 days 01:55:00
2,128,0 days 22:05:00
3,290,5 days 08:00:00
4,401,1 days 12:30:00


In [ ]:
df_clean[df_clean["Train_No"] == 290][
    ["Station_Name", "Arrival_time", "Departure_Time",
     "Arrival_seconds", "Departure_seconds"]
]

,Station_Name,Arrival_time,Departure_Time,Arrival_seconds,Departure_seconds
30,DELHI-SAFDAR,18:30:00,18:30:00,66600,66600
31,GANDHINAGAR,03:45:00,07:45:00,13500,27900
32,JAIPUR JN.,08:00:00,11:55:00,28800,42900
33,DURGAPURA,18:00:00,20:00:00,64800,72000
34,SAWAI MADHOP,04:45:00,08:40:00,17100,31200
35,CHITTAURGARH,00:05:00,02:00:00,300,7200
36,UDAIPUR CITY,08:00:00,11:00:00,28800,39600
37,JAISALMER,09:15:00,13:00:00,33300,46800
38,MANDOR,07:00:00,10:50:00,25200,39000
39,PHULERA JN.,22:00:00,22:55:00,79200,82500


In [ ]:
df_clean[df_clean["Train_No"] == 290][
    ["SN", "Station_Name", "Route_Number",
     "Arrival_time", "Departure_Time", "Distance"]
]

,SN,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
30,1,DELHI-SAFDAR,1,18:30:00,18:30:00,0
31,2,GANDHINAGAR,1,03:45:00,07:45:00,306
32,3,JAIPUR JN.,1,08:00:00,11:55:00,311
33,4,DURGAPURA,1,18:00:00,20:00:00,318
34,5,SAWAI MADHOP,1,04:45:00,08:40:00,442
35,6,CHITTAURGARH,1,00:05:00,02:00:00,707
36,7,UDAIPUR CITY,1,08:00:00,11:00:00,821
37,8,JAISALMER,1,09:15:00,13:00:00,1660
38,9,MANDOR,1,07:00:00,10:50:00,1951
39,10,PHULERA JN.,1,22:00:00,22:55:00,2213


In [ ]:
df_clean[df_clean["Train_No"] == 290][
    ["Train_No", "Station_Name", "Arrival_time",
     "Departure_Time", "Distance"]
].tail(1)

,Train_No,Station_Name,Arrival_time,Departure_Time,Distance
43,290,DELHI-SAFDAR,02:30:00,02:30:00,2694


In [ ]:
arrival_backward_jumps = (
    df_clean["Arrival_seconds"]
    < df_clean.groupby("Train_No")["Arrival_seconds"].shift(1)
)

midnight_jumps_by_train = (
    arrival_backward_jumps
    .groupby(df_clean["Train_No"])
    .sum()
)

midnight_jumps_by_train.describe()

count    11113.000000
mean         0.280122
std          0.563545
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          6.000000
Name: Arrival_seconds, dtype: float64

In [ ]:
(midnight_jumps_by_train > 1).sum()

np.int64(538)

In [ ]:
train_290 = df_clean[df_clean["Train_No"] == 290].copy()

train_290["Previous_Arrival"] = (
    train_290["Arrival_seconds"].shift(1)
)

train_290["Backward_Jump"] = (
    train_290["Arrival_seconds"]
    < train_290["Previous_Arrival"]
)

train_290[
    train_290["Backward_Jump"]
][
    ["SN", "Station_Name", "Arrival_time",
     "Departure_Time", "Previous_Arrival", "Distance"]
]

,SN,Station_Name,Arrival_time,Departure_Time,Previous_Arrival,Distance
31,2,GANDHINAGAR,03:45:00,07:45:00,66600.0,306
34,5,SAWAI MADHOP,04:45:00,08:40:00,64800.0,442
35,6,CHITTAURGARH,00:05:00,02:00:00,17100.0,707
38,9,MANDOR,07:00:00,10:50:00,33300.0,1951
40,11,BHARATPUR JN,05:00:00,08:45:00,79200.0,2455
43,14,DELHI-SAFDAR,02:30:00,02:30:00,68400.0,2694


In [ ]:
train_290["Previous_Departure"] = (
    train_290["Departure_seconds"].shift(1)
)

train_290["Travel_Time_Seconds"] = (
    train_290["Arrival_seconds"]
    - train_290["Previous_Departure"]
)

train_290[
    ["SN", "Station_Name", "Departure_Time",
     "Arrival_time", "Previous_Departure",
     "Travel_Time_Seconds", "Distance"]
]

,SN,Station_Name,Departure_Time,Arrival_time,Previous_Departure,Travel_Time_Seconds,Distance
30,1,DELHI-SAFDAR,18:30:00,18:30:00,NaN,NaN,0
31,2,GANDHINAGAR,07:45:00,03:45:00,66600.0,-53100.0,306
32,3,JAIPUR JN.,11:55:00,08:00:00,27900.0,900.0,311
33,4,DURGAPURA,20:00:00,18:00:00,42900.0,21900.0,318
34,5,SAWAI MADHOP,08:40:00,04:45:00,72000.0,-54900.0,442
35,6,CHITTAURGARH,02:00:00,00:05:00,31200.0,-30900.0,707
36,7,UDAIPUR CITY,11:00:00,08:00:00,7200.0,21600.0,821
37,8,JAISALMER,13:00:00,09:15:00,39600.0,-6300.0,1660
38,9,MANDOR,10:50:00,07:00:00,46800.0,-21600.0,1951
39,10,PHULERA JN.,22:55:00,22:00:00,39000.0,40200.0,2213


In [ ]:
train_290["Next_Arrival_Seconds"] = (
    train_290["Arrival_seconds"].shift(-1)
)

train_290["Current_Departure_Seconds"] = (
    train_290["Departure_seconds"]
)

train_290["Next_Travel_Seconds"] = (
    train_290["Next_Arrival_Seconds"]
    - train_290["Current_Departure_Seconds"]
)

train_290[
    ["SN", "Station_Name", "Departure_Time",
     "Next_Arrival_Seconds", "Next_Travel_Seconds", "Distance"]
]

,SN,Station_Name,Departure_Time,Next_Arrival_Seconds,Next_Travel_Seconds,Distance
30,1,DELHI-SAFDAR,18:30:00,13500.0,-53100.0,0
31,2,GANDHINAGAR,07:45:00,28800.0,900.0,306
32,3,JAIPUR JN.,11:55:00,64800.0,21900.0,311
33,4,DURGAPURA,20:00:00,17100.0,-54900.0,318
34,5,SAWAI MADHOP,08:40:00,300.0,-30900.0,442
35,6,CHITTAURGARH,02:00:00,28800.0,21600.0,707
36,7,UDAIPUR CITY,11:00:00,33300.0,-6300.0,821
37,8,JAISALMER,13:00:00,25200.0,-21600.0,1660
38,9,MANDOR,10:50:00,79200.0,40200.0,1951
39,10,PHULERA JN.,22:55:00,18000.0,-64500.0,2213


In [ ]:
train_290["Adjusted_Next_Arrival"] = train_290["Next_Arrival_Seconds"]

train_290.loc[
    train_290["Adjusted_Next_Arrival"] < train_290["Current_Departure_Seconds"],
    "Adjusted_Next_Arrival"
] += 24 * 60 * 60

train_290[
    ["SN", "Station_Name", "Departure_Time",
     "Next_Arrival_Seconds", "Adjusted_Next_Arrival"]
]

,SN,Station_Name,Departure_Time,Next_Arrival_Seconds,Adjusted_Next_Arrival
30,1,DELHI-SAFDAR,18:30:00,13500.0,99900.0
31,2,GANDHINAGAR,07:45:00,28800.0,28800.0
32,3,JAIPUR JN.,11:55:00,64800.0,64800.0
33,4,DURGAPURA,20:00:00,17100.0,103500.0
34,5,SAWAI MADHOP,08:40:00,300.0,86700.0
35,6,CHITTAURGARH,02:00:00,28800.0,28800.0
36,7,UDAIPUR CITY,11:00:00,33300.0,119700.0
37,8,JAISALMER,13:00:00,25200.0,111600.0
38,9,MANDOR,10:50:00,79200.0,79200.0
39,10,PHULERA JN.,22:55:00,18000.0,104400.0


In [ ]:
train_290["Adjusted_Travel_Seconds"] = (
    train_290["Adjusted_Next_Arrival"]
    - train_290["Current_Departure_Seconds"]
)

train_290[
    ["SN", "Station_Name", "Departure_Time",
     "Adjusted_Next_Arrival", "Adjusted_Travel_Seconds"]
]

,SN,Station_Name,Departure_Time,Adjusted_Next_Arrival,Adjusted_Travel_Seconds
30,1,DELHI-SAFDAR,18:30:00,99900.0,33300.0
31,2,GANDHINAGAR,07:45:00,28800.0,900.0
32,3,JAIPUR JN.,11:55:00,64800.0,21900.0
33,4,DURGAPURA,20:00:00,103500.0,31500.0
34,5,SAWAI MADHOP,08:40:00,86700.0,55500.0
35,6,CHITTAURGARH,02:00:00,28800.0,21600.0
36,7,UDAIPUR CITY,11:00:00,119700.0,80100.0
37,8,JAISALMER,13:00:00,111600.0,64800.0
38,9,MANDOR,10:50:00,79200.0,40200.0
39,10,PHULERA JN.,22:55:00,104400.0,21900.0


In [ ]:
train_290["Adjusted_Travel_Seconds"].describe()

count       13.000000
mean     33230.769231
std      22198.024860
min        900.000000
25%      21900.000000
50%      27000.000000
75%      40200.000000
max      80100.000000
Name: Adjusted_Travel_Seconds, dtype: float64

In [ ]:
df_clean[["Arrival_time", "Departure_Time"]].head(20)

,Arrival_time,Departure_Time
0,00:00:00,10:25:00
1,11:06:00,11:08:00
2,11:28:00,11:30:00
3,12:10:00,00:00:00
4,00:00:00,20:30:00
5,21:04:00,21:06:00
6,21:26:00,21:28:00
7,22:25:00,00:00:00
8,19:40:00,19:40:00
9,20:18:00,20:20:00


In [ ]:
df_clean[df_clean["Train_No"] == 128][
    ["SN", "Station_Name", "Arrival_time",
     "Departure_Time", "Distance"]
]

,SN,Station_Name,Arrival_time,Departure_Time,Distance
8,1,MADGOAN JN.,19:40:00,19:40:00,0
9,2,KARMALI,20:18:00,20:20:00,33
10,3,THIVIM,20:40:00,20:42:00,51
11,4,SAWANTWADI R,21:16:00,21:18:00,83
12,5,KUDAL,21:38:00,21:40:00,104
13,6,SINDHU DURG,21:54:00,21:56:00,114
14,7,KANKAVALI,22:18:00,22:20:00,132
15,8,VAIBHAVWADI,22:40:00,22:42:00,163
16,9,RAJAPUR ROAD,22:56:00,22:58:00,179
17,10,RATNAGIRI,23:52:00,23:57:00,244


In [ ]:
train_data[train_data["Train_No"] == 128][
    ["Train_No", "Start_Departure", "End_Arrival"]
]

,Train_No,Start_Departure,End_Arrival
2,128,19:40:00,17:45:00


In [ ]:
departure_backward_jumps = (
    df_clean["Departure_seconds"]
    < df_clean.groupby("Train_No")["Departure_seconds"].shift(1)
)

departure_jumps_by_train = (
    departure_backward_jumps
    .groupby(df_clean["Train_No"])
    .sum()
)

departure_jumps_by_train.describe()

count    11113.000000
mean         0.455773
std          0.635275
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          6.000000
Name: Departure_seconds, dtype: float64

In [ ]:
same_station_overnight = (
    df_clean["Departure_seconds"] < df_clean["Arrival_seconds"]
)

same_station_overnight.sum()

np.int64(2075)

In [ ]:
df_clean[
    df_clean["Departure_seconds"] < df_clean["Arrival_seconds"]
][
    ["Train_No", "Station_Name",
     "Arrival_time", "Departure_Time", "Distance"]
].head(20)

,Train_No,Station_Name,Arrival_time,Departure_Time,Distance
3,107,MADGOAN JN.,12:10:00,00:00:00,78
7,108,SAWANTWADI R,22:25:00,00:00:00,83
474,2515,HOWRAH JN.,23:55:00,00:15:00,2586
1003,4831,HISAR JN,23:45:00,00:20:00,471
1011,4831,HARIDWAR JN,10:45:00,00:00:00,882
1329,5715,KISHANGANJ,05:10:00,00:00:00,87
1331,5716,NEW JALPAIGU,06:30:00,00:00:00,87
1342,5719,SILIGURI JN.,03:15:00,00:00:00,204
1353,5720,KATIHAR JN.,23:00:00,00:00:00,205
1873,6081,POLLACHI,14:45:00,00:00:00,45


In [ ]:
test_128 = df_clean[df_clean["Train_No"] == 128].copy()

test_128["Arrival_elapsed"] = test_128["Arrival_seconds"].astype(float)
test_128["Departure_elapsed"] = test_128["Departure_seconds"].astype(float)

for i in range(1, len(test_128)):
    while (
        test_128["Arrival_elapsed"].iloc[i]
        < test_128["Departure_elapsed"].iloc[i - 1]
    ):
        test_128.iloc[i, test_128.columns.get_loc("Arrival_elapsed")] += 24 * 60 * 60

    test_128.iloc[i, test_128.columns.get_loc("Departure_elapsed")] = max(
        test_128["Departure_elapsed"].iloc[i],
        test_128["Arrival_elapsed"].iloc[i]
    )

test_128[
    ["Station_Name", "Arrival_time", "Departure_Time",
     "Arrival_elapsed", "Departure_elapsed"]
].tail(8)

,Station_Name,Arrival_time,Departure_Time,Arrival_elapsed,Departure_elapsed
22,ROHA,04:55:00,05:00:00,104100.0,104100.0
23,PANVEL,06:20:00,06:25:00,109200.0,109200.0
24,KARJAT,07:40:00,07:45:00,114000.0,114000.0
25,LONAVLA,08:30:00,08:33:00,117000.0,117000.0
26,PUNE JN.,10:30:00,10:45:00,124200.0,124200.0
27,SATARA,13:42:00,13:45:00,135720.0,135720.0
28,MIRAJ JN.,16:25:00,16:35:00,145500.0,145500.0
29,CHHATRAPATI,17:45:00,17:45:00,150300.0,150300.0


In [ ]:
test_12617 = df_clean[df_clean["Train_No"] == 12617].copy()

test_12617["Arrival_elapsed"] = test_12617["Arrival_seconds"].astype(float)
test_12617["Departure_elapsed"] = test_12617["Departure_seconds"].astype(float)

for i in range(1, len(test_12617)):
    while (
        test_12617["Arrival_elapsed"].iloc[i]
        < test_12617["Departure_elapsed"].iloc[i - 1]
    ):
        test_12617.iloc[
            i, test_12617.columns.get_loc("Arrival_elapsed")
        ] += 24 * 60 * 60

    test_12617.iloc[
        i, test_12617.columns.get_loc("Departure_elapsed")
    ] = max(
        test_12617["Departure_elapsed"].iloc[i],
        test_12617["Arrival_elapsed"].iloc[i]
    )

start = test_12617["Departure_elapsed"].iloc[0]
end = test_12617["Arrival_elapsed"].iloc[-1]

pd.to_timedelta(end - start, unit="s")

Timedelta('2 days 00:00:00')

In [ ]:
def calculate_journey_duration(group):
    group = group.copy()

    arrival = group["Arrival_seconds"].astype(float).to_numpy()
    departure = group["Departure_seconds"].astype(float).to_numpy()

    # First station: journey starts at its departure
    elapsed_arrival = arrival.copy()
    elapsed_departure = departure.copy()

    for i in range(1, len(group)):
        # Move arrival forward until it follows the previous departure
        while elapsed_arrival[i] < elapsed_departure[i - 1]:
            elapsed_arrival[i] += 24 * 60 * 60

        # Departure cannot be earlier than arrival at the same station
        elapsed_departure[i] = max(
            elapsed_departure[i],
            elapsed_arrival[i]
        )

    duration_seconds = (
        elapsed_arrival[-1] - elapsed_departure[0]
    )

    return pd.Series({
        "Journey_Duration": pd.to_timedelta(
            duration_seconds,
            unit="s"
        )
    })


journey_durations = (
    df_clean.groupby("Train_No", sort=False)
    .apply(calculate_journey_duration)
    .reset_index()
)

journey_durations.head(10)

,Train_No,Journey_Duration
0,107,0 days 01:45:00
1,108,0 days 01:55:00
2,128,0 days 22:05:00
3,290,5 days 08:00:00
4,401,1 days 12:30:00
5,421,1 days 09:00:00
6,422,1 days 02:45:00
7,477,2 days 03:10:00
8,502,0 days 23:00:00
9,504,1 days 01:00:00


In [ ]:
journey_durations["Journey_Duration"].describe()

count              11113
mean     0 days 07:13:50
std      0 days 11:07:36
min      0 days 00:05:00
25%      0 days 01:03:00
50%      0 days 02:15:00
75%      0 days 07:30:00
max      5 days 08:00:00
Name: Journey_Duration, dtype: object

In [ ]:
journey_durations.sort_values(
    "Journey_Duration",
    ascending=False
).head(10)

,Train_No,Journey_Duration
3,290,5 days 08:00:00
1935,15906,3 days 10:45:00
1934,15905,3 days 07:40:00
36,2515,3 days 04:55:00
907,12515,3 days 04:35:00
908,12516,3 days 02:45:00
900,12508,3 days 02:45:00
899,12507,3 days 02:05:00
2021,16317,3 days 01:05:00
2118,16688,3 days 00:50:00


In [ ]:
top_longest = journey_durations.sort_values(
    "Journey_Duration",
    ascending=False
).head(10)

top_longest.merge(
    train_data[
        ["Train_No", "Start_Station", "End_Station",
         "Total_Distance", "Number_of_Stops"]
    ],
    on="Train_No",
    how="left"
)

,Train_No,Journey_Duration,Start_Station,End_Station,Total_Distance,Number_of_Stops
0,290,5 days 08:00:00,DELHI-SAFDAR,DELHI-SAFDAR,2694,14
1,15906,3 days 10:45:00,DIBRUGARH,KANNIYAKUMAR,4256,58
2,15905,3 days 07:40:00,KANNIYAKUMAR,DIBRUGARH,4260,57
3,2515,3 days 04:55:00,TRIVANDRUM C,SILCHAR,3939,58
4,12515,3 days 04:35:00,TRIVANDRUM C,SILCHAR,3932,57
5,12516,3 days 02:45:00,SILCHAR,TRIVANDRUM C,3930,56
6,12508,3 days 02:45:00,SILCHAR,TRIVANDRUM C,3930,56
7,12507,3 days 02:05:00,TRIVANDRUM C,SILCHAR,3932,58
8,16317,3 days 01:05:00,KANNIYAKUMAR,SHRI MATA VA,3769,71
9,16688,3 days 00:50:00,SHRI MATA VA,MANGALORE CE,3661,67


In [ ]:
df_clean[df_clean["Train_No"] == 290][
    [
        "Station_Name",
        "Arrival_time",
        "Departure_Time",
        "Distance"
    ]
]

,Station_Name,Arrival_time,Departure_Time,Distance
30,DELHI-SAFDAR,18:30:00,18:30:00,0
31,GANDHINAGAR,03:45:00,07:45:00,306
32,JAIPUR JN.,08:00:00,11:55:00,311
33,DURGAPURA,18:00:00,20:00:00,318
34,SAWAI MADHOP,04:45:00,08:40:00,442
35,CHITTAURGARH,00:05:00,02:00:00,707
36,UDAIPUR CITY,08:00:00,11:00:00,821
37,JAISALMER,09:15:00,13:00:00,1660
38,MANDOR,07:00:00,10:50:00,1951
39,PHULERA JN.,22:00:00,22:55:00,2213


In [ ]:
train_290 = df_clean[df_clean["Train_No"] == 290].copy()

train_290["Arrival_seconds"] = (
    train_290["Arrival_time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

train_290["Departure_seconds"] = (
    train_290["Departure_Time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

train_290["Travel_seconds"] = (
    train_290["Arrival_seconds"].shift(-1)
    - train_290["Departure_seconds"]
)

train_290[
    [
        "Station_Name",
        "Departure_Time",
        "Arrival_time",
        "Travel_seconds"
    ]
]

,Station_Name,Departure_Time,Arrival_time,Travel_seconds
30,DELHI-SAFDAR,18:30:00,18:30:00,-53100.0
31,GANDHINAGAR,07:45:00,03:45:00,900.0
32,JAIPUR JN.,11:55:00,08:00:00,21900.0
33,DURGAPURA,20:00:00,18:00:00,-54900.0
34,SAWAI MADHOP,08:40:00,04:45:00,-30900.0
35,CHITTAURGARH,02:00:00,00:05:00,21600.0
36,UDAIPUR CITY,11:00:00,08:00:00,-6300.0
37,JAISALMER,13:00:00,09:15:00,-21600.0
38,MANDOR,10:50:00,07:00:00,40200.0
39,PHULERA JN.,22:55:00,22:00:00,-64500.0


In [ ]:
train_290 = df_clean[df_clean["Train_No"] == 290].copy()

train_290["Arrival_seconds"] = (
    train_290["Arrival_time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

train_290["Departure_seconds"] = (
    train_290["Departure_Time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

elapsed_arrival = (
    train_290["Arrival_seconds"]
    .astype(float)
    .to_numpy()
    .copy()
)

elapsed_departure = (
    train_290["Departure_seconds"]
    .astype(float)
    .to_numpy()
    .copy()
)

for i in range(1, len(train_290)):
    while elapsed_arrival[i] < elapsed_departure[i - 1]:
        elapsed_arrival[i] += 24 * 60 * 60

    while elapsed_departure[i] < elapsed_arrival[i]:
        elapsed_departure[i] += 24 * 60 * 60

print("Start departure:", elapsed_departure[0])
print("Final arrival:", elapsed_arrival[-1])

print(
    "Journey duration:",
    pd.to_timedelta(
        elapsed_arrival[-1] - elapsed_departure[0],
        unit="s"
    )
)

Start departure: 66600.0
Final arrival: 613800.0
Journey duration: 6 days 08:00:00


In [ ]:
df_clean[
    ["SN", "Train_No", "Station_Name", "Arrival_time", "Departure_Time"]
].head(20)

,SN,Train_No,Station_Name,Arrival_time,Departure_Time
0,1,107,SAWANTWADI R,00:00:00,10:25:00
1,2,107,THIVIM,11:06:00,11:08:00
2,3,107,KARMALI,11:28:00,11:30:00
3,4,107,MADGOAN JN.,12:10:00,00:00:00
4,1,108,MADGOAN JN.,00:00:00,20:30:00
5,2,108,KARMALI,21:04:00,21:06:00
6,3,108,THIVIM,21:26:00,21:28:00
7,4,108,SAWANTWADI R,22:25:00,00:00:00
8,1,128,MADGOAN JN.,19:40:00,19:40:00
9,2,128,KARMALI,20:18:00,20:20:00


In [ ]:
zero_time_counts = pd.DataFrame({
    "Arrival_00:00": [
        (df_clean["Arrival_time"] == pd.to_datetime("00:00:00").time()).sum()
    ],
    "Departure_00:00": [
        (df_clean["Departure_Time"] == pd.to_datetime("00:00:00").time()).sum()
    ]
})

zero_time_counts

,Arrival_00:00,Departure_00:00
0,2003,1970


In [ ]:
first_station = df_clean.groupby("Train_No", sort=False).first()
last_station = df_clean.groupby("Train_No", sort=False).last()

print(
    "First-station Arrival = 00:00:",
    (first_station["Arrival_time"] == pd.to_datetime("00:00:00").time()).sum()
)

print(
    "Last-station Departure = 00:00:",
    (last_station["Departure_Time"] == pd.to_datetime("00:00:00").time()).sum()
)

print(
    "Intermediate Arrival = 00:00:",
    (
        (df_clean["Arrival_time"] == pd.to_datetime("00:00:00").time())
        & (df_clean["SN"] != 1)
    ).sum()
)

print(
    "Intermediate Departure = 00:00:",
    (
        (df_clean["Departure_Time"] == pd.to_datetime("00:00:00").time())
        & (
            df_clean["SN"]
            != df_clean.groupby("Train_No")["SN"].transform("max")
        )
    ).sum()
)

First-station Arrival = 00:00: 1951
Last-station Departure = 00:00: 1955
Intermediate Arrival = 00:00: 52
Intermediate Departure = 00:00: 15


In [ ]:
train_290 = df_clean[df_clean["Train_No"] == 290].copy()

train_290["Arrival_seconds"] = (
    train_290["Arrival_time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

train_290["Departure_seconds"] = (
    train_290["Departure_Time"].apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
)

elapsed_departure = train_290["Departure_seconds"].astype(float).to_numpy().copy()
elapsed_arrival = train_290["Arrival_seconds"].astype(float).to_numpy().copy()

# Journey starts at the first station's departure
elapsed_departure[0] = train_290["Departure_seconds"].iloc[0]

for i in range(1, len(train_290)):

    # Move arrival into the next chronological day when necessary
    while elapsed_arrival[i] < elapsed_departure[i - 1]:
        elapsed_arrival[i] += 24 * 60 * 60

    # Move departure into the same or following day when necessary
    while elapsed_departure[i] < elapsed_arrival[i]:
        elapsed_departure[i] += 24 * 60 * 60

journey_duration_seconds = (
    elapsed_arrival[-1] - elapsed_departure[0]
)

print(
    "Train 290 Journey Duration:",
    pd.to_timedelta(journey_duration_seconds, unit="s")
)

Train 290 Journey Duration: 6 days 08:00:00


In [ ]:
def count_time_wraps(group):
    arrival = (
        group["Arrival_time"]
        .apply(lambda x: x.hour * 3600 + x.minute * 60 + x.second)
        .to_numpy()
    )

    departure = (
        group["Departure_Time"]
        .apply(lambda x: x.hour * 3600 + x.minute * 60 + x.second)
        .to_numpy()
    )

    arrival_wraps = (arrival[1:] < departure[:-1]).sum()
    departure_wraps = (departure < arrival).sum()

    return pd.Series({
        "Arrival_Wraps": arrival_wraps,
        "Departure_Before_Arrival": departure_wraps
    })


time_checks = (
    df_clean.groupby("Train_No", sort=False)
    .apply(count_time_wraps)
    .reset_index()
)

time_checks.describe()

,Train_No,Arrival_Wraps,Departure_Before_Arrival
count,11113.000000,11113.000000,11113.000000
mean,49190.570413,0.271484,0.186718
std,28515.986645,0.555094,0.392235
min,107.000000,0.000000,0.000000
25%,22607.000000,0.000000,0.000000
50%,47174.000000,0.000000,0.000000
75%,68012.000000,0.000000,0.000000
max,99908.000000,7.000000,2.000000


In [ ]:
duration_check = journey_durations.merge(
    train_data[
        [
            "Train_No",
            "Total_Distance",
            "Number_of_Stops"
        ]
    ],
    on="Train_No",
    how="left"
)

duration_check["Duration_hours"] = (
    duration_check["Journey_Duration"].dt.total_seconds() / 3600
)

duration_check["Distance_per_hour"] = (
    duration_check["Total_Distance"]
    / duration_check["Duration_hours"]
)

duration_check[
    [
        "Train_No",
        "Journey_Duration",
        "Total_Distance",
        "Number_of_Stops",
        "Duration_hours",
        "Distance_per_hour"
    ]
].sort_values(
    "Distance_per_hour",
    ascending=False
).head(10)

,Train_No,Journey_Duration,Total_Distance,Number_of_Stops,Duration_hours,Distance_per_hour
1327,12978,0 days 06:00:00,2560,7,6.000000,426.666667
11,557,0 days 11:30:00,1301,2,11.500000,113.130435
480,12050,0 days 01:40:00,183,2,1.666667,109.800000
9360,82665,0 days 06:09:00,668,15,6.150000,108.617886
479,12049,0 days 01:40:00,181,2,1.666667,108.600000
9361,82666,0 days 06:08:00,663,15,6.133333,108.097826
258,9004,0 days 13:55:00,1358,5,13.916667,97.580838
257,9003,0 days 13:55:00,1355,5,13.916667,97.365269
1302,12951,0 days 15:35:00,1373,8,15.583333,88.106952
464,12034,0 days 04:55:00,430,4,4.916667,87.457627


In [ ]:
df_clean[df_clean["Train_No"] == 1327][
    [
        "SN",
        "Station_Name",
        "Arrival_time",
        "Departure_Time",
        "Distance"
    ]
]

,SN,Station_Name,Arrival_time,Departure_Time,Distance


In [ ]:
duration_check.loc[
    duration_check["Train_No"] == 12978,
    [
        "Train_No",
        "Journey_Duration",
        "Total_Distance",
        "Number_of_Stops",
        "Duration_hours",
        "Distance_per_hour"
    ]
]

,Train_No,Journey_Duration,Total_Distance,Number_of_Stops,Duration_hours,Distance_per_hour
1327,12978,0 days 06:00:00,2560,7,6.0,426.666667


In [ ]:
df_clean[df_clean["Train_No"] == 12978][
    [
        "SN",
        "Station_Name",
        "Arrival_time",
        "Departure_Time",
        "Distance"
    ]
].to_string(index=False)

' SN Station_Name Arrival_time Departure_Time  Distance\n 26       KANNUR     21:15:00       21:20:00      2277\n 27      CALICUT     22:35:00       22:40:00      2366\n 28        TIRUR     23:23:00       23:25:00      2407\n 29 SHORANUR JN.     00:20:00       00:25:00      2452\n 30      TRICHUR     01:30:00       01:32:00      2486\n 31       ALWAYE     02:23:00       02:25:00      2541\n 32 ERNAKULAM. J     03:20:00       03:20:00      2560'

In [ ]:
df_clean[df_clean["Train_No"] == 12978][
    [
        "SN",
        "Station_Name",
        "Arrival_time",
        "Departure_Time",
        "Distance"
    ]
].to_string(index=False)

' SN Station_Name Arrival_time Departure_Time  Distance\n 26       KANNUR     21:15:00       21:20:00      2277\n 27      CALICUT     22:35:00       22:40:00      2366\n 28        TIRUR     23:23:00       23:25:00      2407\n 29 SHORANUR JN.     00:20:00       00:25:00      2452\n 30      TRICHUR     01:30:00       01:32:00      2486\n 31       ALWAYE     02:23:00       02:25:00      2541\n 32 ERNAKULAM. J     03:20:00       03:20:00      2560'

In [ ]:
train_12978 = df_clean[df_clean["Train_No"] == 12978].copy()

print("Number of records:", len(train_12978))
print("Minimum SN:", train_12978["SN"].min())
print("Maximum SN:", train_12978["SN"].max())
print("First distance:", train_12978["Distance"].iloc[0])
print("Last distance:", train_12978["Distance"].iloc[-1])

Number of records: 7
Minimum SN: 26
Maximum SN: 32
First distance: 2277
Last distance: 2560


In [ ]:
train_sn_range = df_clean.groupby("Train_No")["SN"].agg(
    First_SN="min",
    Last_SN="max",
    Number_of_Records="count"
).reset_index()

print(
    "Trains starting at SN = 1:",
    (train_sn_range["First_SN"] == 1).sum()
)

print(
    "Trains NOT starting at SN = 1:",
    (train_sn_range["First_SN"] != 1).sum()
)

print(
    "Percentage of trains NOT starting at SN = 1:",
    round(
        (train_sn_range["First_SN"] != 1).mean() * 100,
        2
    ),
    "%"
)

Trains starting at SN = 1: 11112
Trains NOT starting at SN = 1: 1
Percentage of trains NOT starting at SN = 1: 0.01 %


In [ ]:
complete_trains = train_sn_range[
    train_sn_range["First_SN"] == 1
]["Train_No"]

print("Complete trains:", len(complete_trains))

print(
    "Train 12978 included:",
    12978 in complete_trains.values
)

Complete trains: 11112
Train 12978 included: False


In [ ]:
complete_train_numbers = train_sn_range.loc[
    train_sn_range["First_SN"] == 1,
    "Train_No"
]

print("Complete trains:", len(complete_train_numbers))
print("Unique complete trains:", complete_train_numbers.nunique())

Complete trains: 11112
Unique complete trains: 11112


In [ ]:
df_clean[df_clean["Train_No"] == 128][
    [
        "SN",
        "Station_Name",
        "Arrival_time",
        "Departure_Time",
        "Distance"
    ]
].to_string(index=False)

' SN Station_Name Arrival_time Departure_Time  Distance\n  1  MADGOAN JN.     19:40:00       19:40:00         0\n  2      KARMALI     20:18:00       20:20:00        33\n  3       THIVIM     20:40:00       20:42:00        51\n  4 SAWANTWADI R     21:16:00       21:18:00        83\n  5        KUDAL     21:38:00       21:40:00       104\n  6  SINDHU DURG     21:54:00       21:56:00       114\n  7    KANKAVALI     22:18:00       22:20:00       132\n  8  VAIBHAVWADI     22:40:00       22:42:00       163\n  9 RAJAPUR ROAD     22:56:00       22:58:00       179\n 10    RATNAGIRI     23:52:00       23:57:00       244\n 11  SANGMESHWAR     00:34:00       00:36:00       280\n 12      CHIPLUN     01:10:00       01:12:00       322\n 13         KHED     01:52:00       01:54:00       352\n 14      MANGAON     04:00:00       04:02:00       422\n 15         ROHA     04:55:00       05:00:00       455\n 16       PANVEL     06:20:00       06:25:00       532\n 17       KARJAT     07:40:00       07:45:00   

In [ ]:
journey_durations[
    journey_durations["Train_No"] == 128
]

,Train_No,Journey_Duration
2,128,0 days 22:05:00


In [ ]:
multiple_wrap_trains = time_checks[
    (time_checks["Train_No"].isin(complete_train_numbers))
    & (time_checks["Arrival_Wraps"] > 1)
]

print("Complete trains with more than 1 arrival wrap:",
      len(multiple_wrap_trains))

print("Percentage:",
      round(
          len(multiple_wrap_trains) / len(complete_train_numbers) * 100,
          2
      ),
      "%"
)

Complete trains with more than 1 arrival wrap: 517
Percentage: 4.65 %


In [ ]:
print(
    "Maximum arrival wraps among complete trains:",
    multiple_wrap_trains["Arrival_Wraps"].max()
)

print("\nNumber of trains by arrival-wrap count:")
print(
    time_checks[
        time_checks["Train_No"].isin(complete_train_numbers)
    ]["Arrival_Wraps"]
    .value_counts()
    .sort_index()
)

Maximum arrival wraps among complete trains: 7

Number of trains by arrival-wrap count:
Arrival_Wraps
0    8655
1    1940
2     481
3      33
4       2
7       1
Name: count, dtype: int64


In [ ]:
time_checks[
    time_checks["Arrival_Wraps"] == 7
]

,Train_No,Arrival_Wraps,Departure_Before_Arrival
3,290,7,0


In [ ]:
duration_hours = (
    journey_durations["Journey_Duration"]
    .dt.total_seconds() / 3600
)

print("Journeys longer than 4 days:",
      (duration_hours > 96).sum())

print("Journeys longer than 3 days:",
      (duration_hours > 72).sum())

print("Journeys longer than 2 days:",
      (duration_hours > 48).sum())

print("Maximum duration:",
      journey_durations["Journey_Duration"].max())

Journeys longer than 4 days: 1
Journeys longer than 3 days: 10
Journeys longer than 2 days: 138
Maximum duration: 5 days 08:00:00


In [ ]:
duration_seconds = (
    journey_durations["Journey_Duration"]
    .dt.total_seconds()
)

print("Zero-duration trains:",
      (duration_seconds == 0).sum())

print("Negative-duration trains:",
      (duration_seconds < 0).sum())

print("Minimum duration:",
      journey_durations["Journey_Duration"].min())

Zero-duration trains: 0
Negative-duration trains: 0
Minimum duration: 0 days 00:05:00


In [ ]:
complete_journey_durations = journey_durations[
    journey_durations["Train_No"].isin(complete_train_numbers)
].copy()

print("Complete train duration records:",
      len(complete_journey_durations))

print("Unique trains:",
      complete_journey_durations["Train_No"].nunique())

print("Missing journey durations:",
      complete_journey_durations["Journey_Duration"].isna().sum())

Complete train duration records: 11112
Unique trains: 11112
Missing journey durations: 0


In [ ]:
train_data_final = train_data[
    train_data["Train_No"].isin(complete_train_numbers)
].copy()

train_data_final = train_data_final.merge(
    complete_journey_durations,
    on="Train_No",
    how="left"
)

print("Rows:", len(train_data_final))
print("Columns:", train_data_final.columns.tolist())
print("\nMissing values:")
print(train_data_final.isna().sum())

Rows: 11112
Columns: ['Train_No', 'Start_Station', 'End_Station', 'Start_Departure', 'End_Arrival', 'Total_Distance', 'Number_of_Stops', 'Journey_Duration']

Missing values:
Train_No            0
Start_Station       0
End_Station         0
Start_Departure     0
End_Arrival         0
Total_Distance      0
Number_of_Stops     0
Journey_Duration    0
dtype: int64


In [ ]:
excluded_trains = train_sn_range[
    train_sn_range["First_SN"] != 1
].copy()

excluded_trains

,Train_No,First_SN,Last_SN,Number_of_Records
1327,12978,26,32,7


In [ ]:
train_data_final.to_csv(
    "../data/train_level2_cleaned.csv",
    index=False
)

print("Level 2 dataset saved successfully.")

Level 2 dataset saved successfully.


In [ ]:
level2 = pd.read_csv("../data/train_level2_cleaned.csv")

print("Shape:", level2.shape)

print("\nColumns:")
print(level2.columns.tolist())

print("\nMissing values:")
print(level2.isna().sum())

print("\nDuplicate rows:", level2.duplicated().sum())

print("\nDuplicate Train_No:",
      level2["Train_No"].duplicated().sum())

Shape: (11112, 8)

Columns:
['Train_No', 'Start_Station', 'End_Station', 'Start_Departure', 'End_Arrival', 'Total_Distance', 'Number_of_Stops', 'Journey_Duration']

Missing values:
Train_No            0
Start_Station       0
End_Station         0
Start_Departure     0
End_Arrival         0
Total_Distance      0
Number_of_Stops     0
Journey_Duration    0
dtype: int64

Duplicate rows: 0

Duplicate Train_No: 0
